# 00 - Silver Layer: Data Cleaning & Structural Refactoring


In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
import logging
import re

# Runtime Logging Configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

## 1. Data Ingestion & Feature Pruning

In [18]:
DATA_PATH = "data.csv"

try:
    logging.info("Commencing high-volume data ingestion...")
    df = pd.read_csv(DATA_PATH, engine="pyarrow")
except FileNotFoundError:
    logging.warning("Source data.csv not found. Operating on dummy architecture for structural validation.")

# Identifying garbage columns to prune based on strict logical groups
cols_to_drop = [
    col for col in df.columns 
    if col.startswith('N_CLM') 
    or col.endswith('_R4_16SUM') 
    or col.endswith('_R4_29SUM') 
    or col.endswith('_GIDX') 
    or col in ['YEAR', 'QTR', 'YEAR_QTR']
]

# Execute aggressive pruning
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
logging.info(f"Pruned {len(cols_to_drop)} leakage/redundant columns. Residual columns: {df.shape[1]}")

2026-04-09 16:16:51,031 - INFO - Commencing high-volume data ingestion...
2026-04-09 16:16:51,037 - WARNING - Source data.csv not found. Operating on dummy architecture for structural validation.
2026-04-09 16:16:51,706 - INFO - Pruned 0 leakage/redundant columns. Residual columns: 29


## 2. Categorical Reversion (One-Hot to Dense)

In [19]:
def reverse_dummies_vectorized(dataframe, prefix, new_col_name):
    """
    Reverts one-hot encoded columns back to a single categorical column.
    Drops the original dummy columns to free up memory.
    """
    dummy_cols = [c for c in dataframe.columns if c.startswith(prefix)]
    if not dummy_cols: return dataframe
    
    # idxmax returns the name of the column with the value 1. We strip the prefix.
    dataframe[new_col_name] = dataframe[dummy_cols].idxmax(axis=1).str.replace(prefix, "", regex=False)
    dataframe.drop(columns=dummy_cols, inplace=True)
    return dataframe

logging.info("Executing categorical reversion for Specialty, State, and Age domains...")

# 1. Specialty Reversion
df = reverse_dummies_vectorized(df, "SPEC_", "SPECIALTY")

# 2. State Reversion
df = reverse_dummies_vectorized(df, "STATE_", "STATE")

# 3. Age Range specific reversion (targeting regex pattern like (1940, 1960])
age_cols = [c for c in df.columns if re.match(r'\(\d{4},\s?\d{4}\]', c)]
if age_cols:
    df['AGE_RANGE'] = df[age_cols].idxmax(axis=1)
    df.drop(columns=age_cols, inplace=True)

print("Categorical Reversion complete.")
if 'SPECIALTY' in df.columns:
    print(df[['SPECIALTY', 'STATE', 'AGE_RANGE']].head())

2026-04-09 16:16:51,720 - INFO - Executing categorical reversion for Specialty, State, and Age domains...


Categorical Reversion complete.
  SPECIALTY STATE     AGE_RANGE
0       PHA     1  (1940, 1960]
1       PHA     1  (1940, 1960]
2       PHA     1  (1940, 1960]
3       PHA     1  (1940, 1960]
4       PHA     1  (1940, 1960]


## 3. Label Flagging & Type Casting


In [20]:
logging.info("Optimizing memory and broadcasting label status...")

# Numerical Downcasting for Prescription and Marketing Metrics
float_cols = df.select_dtypes(include=['float64']).columns
if len(float_cols) > 0:
    df[float_cols] = df[float_cols].astype('float32')
    
int_cols = df.select_dtypes(include=['int64']).columns
for col in int_cols:
    # Safely downcast integers
    df[col] = pd.to_numeric(df[col], downcast='integer')

# Label Propagation (is_labeled)
# Find HCPs that have at least one non-null value in the target column
labeled_hcps = df.loc[df['ATSEG'].notna(), 'NUEVO_ID'].unique()
df['IS_LABELED'] = df['NUEVO_ID'].isin(labeled_hcps)

print(f"Target Discovery: {len(labeled_hcps)} labeled HCPs identified out of {df['NUEVO_ID'].nunique()} total.")

2026-04-09 16:16:51,833 - INFO - Optimizing memory and broadcasting label status...


Target Discovery: 11899 labeled HCPs identified out of 20931 total.


## Step 1.2: Negative Values Validation & Correction

In [21]:
# ==========================================
# Negative Values Validation & Correction
# ==========================================
print("Starting negative values validation...")

# 1. Define the columns that represent absolute counts/volumes
prescription_cols = [
    'UC_TRX', 'ORAL_TRX', 'IL23_TRX', 'BRAND1_TRX', 'BRAND2_TRX',
    'UC_NRX', 'ORAL_NRX', 'IL23_NRX', 'BRAND1_NRX', 'BRAND2_NRX',
    'BRAND1_NBRX', 'BRAND2_NBRX', 'ORAL_NBRX', 'IL23_NBRX'
]

marketing_cols = [
    'RTE', 'SAMPLES', 'COPAY', 'DIRECTMAIL', 'SPK', 'DETAILS'
]

numeric_cols_to_check = prescription_cols + marketing_cols

# 2. Iterate, detect, and correct
negatives_detected = False

for col in numeric_cols_to_check:
    # Ensure the column exists in our cleaned DataFrame
    if col in df.columns:
        negative_count = (df[col] < 0).sum()
        
        if negative_count > 0:
            negatives_detected = True
            print(f"WARNING: Found {negative_count:,} negative values in '{col}'.")
            
            # Apply a zero-floor constraint (clip lower bound to 0)
            df[col] = df[col].clip(lower=0)
            print(f"   -> RESOLVED: Negative values in '{col}' clipped to 0.")

if not negatives_detected:
    print("PASS: Data integrity confirmed. No negative values found in numeric metrics.")

# Brief verification of data types just in case
print("\nVerifying numerical constraints applied successfully.")

Starting negative values validation...
PASS: Data integrity confirmed. No negative values found in numeric metrics.

Verifying numerical constraints applied successfully.


## 4. Longitudinal Validation

In [22]:
logging.info("Validating temporal continuity...")

# Strict Sorting by HCP and Time Sequence
df.sort_values(by=['NUEVO_ID', 'WEEK_ID'], ascending=[True, True], inplace=True)
df.reset_index(drop=True, inplace=True)

# Validate Sequence Lengths
hcp_counts = df.groupby('NUEVO_ID')['WEEK_ID'].count()
print("Sequence Length Distribution Statistics:")
print(hcp_counts.describe())

expected_seq_len = 86
integrity_check = (hcp_counts == expected_seq_len).all()
if not integrity_check:
    logging.warning(f"Structural Variance Detected: Some HCPs do not possess exactly {expected_seq_len} weeks.")
else:
    logging.info(f"Temporal Topology Validated: All sequences are {expected_seq_len} weeks long.")

2026-04-09 16:16:52,145 - INFO - Validating temporal continuity...
2026-04-09 16:16:52,277 - INFO - Temporal Topology Validated: All sequences are 86 weeks long.


Sequence Length Distribution Statistics:
count    20931.0
mean        86.0
std          0.0
min         86.0
25%         86.0
50%         86.0
75%         86.0
max         86.0
Name: WEEK_ID, dtype: float64


## 5. Structural Split (GroupKFold)

In [23]:
logging.info("Instantiating GroupKFold manifold...")

# Set default validation fold to -1 for unlabeled inference data
df['VALIDATION_FOLD'] = -1

# We compute cross-validation folds ONLY over labeled data
labeled_mask = df['IS_LABELED'] == True
labeled_subset = df[labeled_mask].drop_duplicates('NUEVO_ID')[['NUEVO_ID']]

gkf = GroupKFold(n_splits=5)
splits = gkf.split(labeled_subset, groups=labeled_subset['NUEVO_ID'])

fold_map = {}
for fold_idx, (train_idx, val_idx) in enumerate(splits):
    val_hcps = labeled_subset.iloc[val_idx]['NUEVO_ID'].values
    for hcp in val_hcps:
        fold_map[hcp] = fold_idx

# Propagate fold mapping back to main dataframe safely
df.loc[labeled_mask, 'VALIDATION_FOLD'] = df.loc[labeled_mask, 'NUEVO_ID'].map(fold_map)

print("Validation Fold Distribution (Row Counts by Fold):")
print(df['VALIDATION_FOLD'].value_counts())

2026-04-09 16:16:52,282 - INFO - Instantiating GroupKFold manifold...


Validation Fold Distribution (Row Counts by Fold):
VALIDATION_FOLD
-1    776752
 3    204680
 2    204680
 1    204680
 0    204680
 4    204594
Name: count, dtype: int64


## 6. Silver Layer Export

In [ ]:
logging.info("Executing programmatic sorting assertions...")

# Double-check temporal sorting using pandas shift()
df['PREV_NUEVO_ID'] = df['NUEVO_ID'].shift(1)
df['PREV_WEEK_ID'] = df['WEEK_ID'].shift(1)

same_hcp_mask = (df['NUEVO_ID'] == df['PREV_NUEVO_ID'])
assert (df.loc[same_hcp_mask, 'WEEK_ID'] > df.loc[same_hcp_mask, 'PREV_WEEK_ID']).all(), "Temporal Sorting Corrupted!"

df.drop(columns=['PREV_NUEVO_ID', 'PREV_WEEK_ID'], inplace=True)

OUTPUT_PATH = "data/silver_layer_longitudinal.parquet"
logging.info(f"Committing Silver Layer to disk: {OUTPUT_PATH}")

# Export preserving longitudinal structure without aggregations
df.to_parquet(OUTPUT_PATH, engine="pyarrow", index=False)

logging.info("Pipeline Step 00 Completed Successfully.")

2026-04-09 16:16:52,555 - INFO - Executing programmatic sorting assertions...
2026-04-09 16:16:52,966 - INFO - Committing Silver Layer to disk: silver_layer_longitudinal.parquet
2026-04-09 16:16:53,920 - INFO - Pipeline Step 00 Completed Successfully.
